In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
# Load environment variables from .env file
load_dotenv()

# Get the API key
api_key = os.getenv("OPENAI_API_KEY")

# Use it with OpenAI
client = OpenAI(api_key=api_key)


def health_chatbot(user_query):
    # Manual Safety Filter
    dangerous_topics = [
        "suicide", "overdose", "kill yourself", "self-harm",
        "how to make", "how to create", "how to obtain"
    ]
    question = user_query.lower()
    for i in dangerous_topics:
        if i in question:
            return "⚠️ I cannot provide information about this topic for safety reasons. Please consult a medical professional immediately."

    
    # 2. Prompt Engineering: The System Message
    system_prompt = (
        "You are a helpful, friendly medical assistant. "
        "Your goal is to provide general health information, not medical advice. "
        "Rules: \n"
        "1. Be empathetic and clear.\n"
        "2. If a user asks for a diagnosis or specific dosage, provide general information "
        "and strongly advise them to consult a doctor.\n"
        "3. If the query sounds like an emergency, tell them to call emergency services immediately.\n"
        "4. Always end with a medical disclaimer."
    )

    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_query}
            ],
            temperature=0.7,
            max_tokens=500
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

# 3. Simple Chat Loop
print("HealthBot: Hello! I can provide general health info. (Type 'quit' to exit)")
while True:
    user_input = input("You: ")
    if user_input.lower() == 'quit':
        break
    
    result = health_chatbot(user_input)
    print(f"\nHealthBot: {result}\n")